# Lab 2.1 · The concurrency cap

Day 2 · Session 1 (Continuous batching and admission) · 25 min.

Yamen hands you two deployments of Musamir that miss the objective at a moderate load of 6 concurrent visitors.
One queues, one preempts under cache pressure. You read each one's engine metrics, vary the sequence cap,
plot tail latency against throughput, and write a diagnosis per fault with its evidence and a rerun that
meets the objective (90% of requests with TTFT at most 2.0 s, TPOT at most 150 ms and no stall over 1.0 s).

**AI tutor allowed**: it may explain engine options; it does not choose settings.

**Starts from** `data/deployments/fault_queueing.json` and `fault_preemption.json` (seeded faults 1 and 2).

**Tasks**
- [ ] 1 · Run the queueing deployment and read its metrics (5 min)
- [ ] 2 · Vary the sequence cap over five values (6 min)
- [ ] 3 · Plot p90 TTFT against throughput (5 min)
- [ ] 4 · Push into cache pressure and count preemptions (5 min)
- [ ] 5 · Choose the cap and write both diagnoses (4 min)
- [ ] ✱ Stretch · Hold the cap high and limit admission with the token budget instead

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/SDAIAAcademy/ae-401.git"
    if not Path("/content/ae-401").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/ae-401"], check=True)
    os.chdir("/content/ae-401/labs")
else:
    while not Path("_tools").exists() and Path.cwd() != Path.cwd().parent:
        os.chdir("..")  # solutions/ notebooks run from labs/
sys.path.insert(0, os.getcwd())

import pandas as pd
import servelab as S

OBJ = S.objectives()
USERS = 6
N = S.pick(240, 140)
q_cfg = S.Config.load("fault_queueing")
p_cfg = S.Config.load("fault_preemption")
print({k: getattr(q_cfg, k) for k in ("max_num_seqs", "gpu_memory_utilization", "enable_chunked_prefill", "max_num_batched_tokens")})

## 1 · Run the queueing deployment and read its metrics (5 min)

An engine exports gauges: sequences running, requests waiting, cache usage, preemptions so far (in vLLM:
`vllm:num_requests_running`, `vllm:num_requests_waiting`, `vllm:kv_cache_usage_perc`). The simulator keeps the
same gauges in `run.engine_log()`.

✏️ **Exercise** · `signature(run)` returns the mean of `running`, `waiting` and `kv_cache_usage` over the log,
and the preemptions per 100 requests (`run.engines[0].preemptions`, `len(run.all)`).

In [ ]:
def signature(run):
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
q_run = S.run_closed(q_cfg, USERS, n_requests=N, seed=1)
try:
    q_sig = signature(q_run)
except NotImplementedError:
    q_sig = S.engine_signature(q_run)
print("summary  :", {k: q_run.summary()[k] for k in ("goodput_rps", "attainment", "ttft_p90", "tpot_p90")})
print("signature:", q_sig)
S.check(q_sig["running"] >= q_cfg.max_num_seqs * 0.9 and q_sig["waiting"] > 1 and q_sig["kv_cache_usage"] < 0.5,
        "running at the cap, a queue waiting, the cache half empty: queueing")

## 2 · Vary the sequence cap over five values (6 min)

✏️ **Exercise** · For caps 2, 4, 6, 8 and 12, run the same load and collect throughput, goodput, attainment,
p90 TTFT, p90 TPOT and preemptions in `rows`.

In [ ]:
CAPS = (2, 4, 6, 8, 12)
rows = []
try:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")
except NotImplementedError:
    rows = S.reference("d2_1")["caps"]
caps = pd.DataFrame(rows)
print(caps.to_string(index=False))
S.save_rows(rows, "D2_1", "cap_sweep")

## 3 · Plot p90 TTFT against throughput (5 min)

In [ ]:
plt = S.plt()
fig, ax = plt.subplots()
ax.plot(caps.throughput_rps, caps.ttft_p90, marker="o")
for _, r in caps.iterrows():
    ax.annotate(f"cap {int(r.cap)}", (r.throughput_rps, r.ttft_p90), textcoords="offset points", xytext=(5, 5))
ax.axhline(OBJ["ttft_s"], ls="--", c="grey")
ax.set(xlabel="throughput (requests per second)", ylabel="p90 TTFT (s)", yscale="log", title=f"{USERS} users, cap varied")
S.show(fig, "ttft_vs_throughput", "D2_1")

🔍 **Look at** the step from cap 2 to cap 6: throughput more than doubles and the tail falls by an order of
magnitude. Past the number of users, a higher cap changes nothing at this load.

## 4 · Push into cache pressure and count preemptions (5 min)

The second deployment has a cap of 48 and leaves less memory to the engine (`gpu_memory_utilization` 0.82).

In [ ]:
p_run = S.run_closed(p_cfg, USERS, n_requests=N, seed=1)
try:
    p_sig = signature(p_run)
except NotImplementedError:
    p_sig = S.engine_signature(p_run)
print(S.start_log(p_cfg).splitlines()[2])
print("summary  :", {k: p_run.summary()[k] for k in ("goodput_rps", "attainment", "ttft_p90", "tpot_p90", "preemptions")})
print("signature:", p_sig)
S.check(p_sig["kv_cache_usage"] > 0.85 and p_sig["preemptions_per_100"] > 5,
        "the cache nearly full and sequences preempted: cache pressure")

## 5 · Choose the cap and write both diagnoses (4 min)

✏️ **Exercise** · `diagnose(sig)` returns `"queueing"` when the engine runs at its cap with a queue and room in the
cache, `"preemption"` when the cache is nearly full and sequences are preempted, otherwise `"ok"`. Then set
`fixed` to a configuration that meets the objective at this load and rerun it.

In [ ]:
def diagnose(sig):
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")


try:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")
except NotImplementedError:
    fixed = S.Config.load("tuned_day2").but(layout="deployed", cache_salt="none")
try:
    verdicts = {"fault_queueing": diagnose(q_sig), "fault_preemption": diagnose(p_sig)}
except NotImplementedError:
    verdicts = {"fault_queueing": "queueing", "fault_preemption": "preemption"}
f_run = S.run_closed(fixed, USERS, n_requests=N, seed=1)
f_sum, f_sig = f_run.summary(), S.engine_signature(f_run)
print(verdicts)
print("rerun:", {k: f_sum[k] for k in ("goodput_rps", "attainment", "ttft_p90", "tpot_p90", "preemptions")}, f_sig)
S.check(verdicts == {"fault_queueing": "queueing", "fault_preemption": "preemption"}, "both faults named from their metrics")
S.check(f_sum["attainment"] >= OBJ["attainment_target"], "the rerun meets the objective at 6 users")
S.save_rows([{"deployment": k, "diagnosis": v} for k, v in verdicts.items()], "D2_1", "diagnoses")

✱ **Stretch** · Keep the cap at 48 but make each step's token budget small (`max_num_batched_tokens=512`):
does limiting admission by tokens protect the cache as well as limiting it by sequences?

In [ ]:
st = S.run_closed(p_cfg.but(gpu_memory_utilization=0.90, max_num_batched_tokens=512), USERS, n_requests=N, seed=1).summary()
print({k: st[k] for k in ("attainment", "ttft_p90", "tpot_p90", "preemptions")})

## Check yourself

1. Two deployments both have a p90 TTFT over 20 s. How do you tell queueing from preemption?
   <details><summary>Answer</summary>Queueing: running at the cap, a queue, the cache with room. Preemption: the cache nearly full and the preemption counter rising. The fixes are opposite: raise the cap, or give the cache room and lower the cap.</details>
2. Why does a cap above 6 change nothing at 6 users?
   <details><summary>Answer</summary>A closed loop of 6 users never has more than 6 requests in the engine.</details>
3. Why is a preempted request worse than a queued one?
   <details><summary>Answer</summary>Its work is thrown away and recomputed, so the engine spends capacity twice and the visitor sees a stall in the middle of an answer.</details>

**Carry forward** · `artifacts/D2_1/diagnoses.csv`: two of the three diagnoses of the outcome; Lab 2.2 adds the third.